In [ ]:
# ============================================================
# Task 1: News Topic Classifier Using BERT
# Dataset: AG News from Hugging Face
# Model: bert-base-uncased
# Evaluation: Accuracy + F1-score
# Deployment: Professional Gradio UI
# Jupyter Fixed Version
# ============================================================

import sys
import subprocess
import os
import warnings
import logging
import contextlib

# ============================================================
# 1. Clean Warnings and Widget Errors
# ============================================================

warnings.filterwarnings("ignore")

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TQDM_DISABLE"] = "1"

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("datasets").setLevel(logging.ERROR)

# ============================================================
# 2. Install Required Packages
# ============================================================

packages = {
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
    "sklearn": "scikit-learn",
    "gradio": "gradio",
    "numpy": "numpy",
    "accelerate": "accelerate"
}

for import_name, install_name in packages.items():
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            install_name
        ])

# ============================================================
# 3. Import Libraries
# ============================================================

import torch
import numpy as np
import gradio as gr

from datasets import load_dataset, disable_progress_bar
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from transformers.utils import logging as hf_logging

disable_progress_bar()
hf_logging.set_verbosity_error()

try:
    gr.close_all()
except:
    pass

# ============================================================
# 4. Load AG News Dataset
# ============================================================

print("Loading AG News dataset...")

dataset = load_dataset("ag_news")

id_to_label = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

label_to_id = {
    "World": 0,
    "Sports": 1,
    "Business": 2,
    "Sci/Tech": 3
}

print("Dataset loaded successfully")
print(dataset)
print("Sample:", dataset["train"][0])

# ============================================================
# 5. Small Dataset for Fast Jupyter Training
# ============================================================

# CPU par fast run ke liye small dataset rakha hai.
# Better accuracy ke liye 800 ko 2000 ya 5000 kar sakte ho.

TRAIN_SAMPLES = 800
TEST_SAMPLES = 200

train_dataset = dataset["train"].shuffle(seed=42).select(range(TRAIN_SAMPLES))
test_dataset = dataset["test"].shuffle(seed=42).select(range(TEST_SAMPLES))

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

# ============================================================
# 6. Tokenization and Preprocessing
# ============================================================

MODEL_NAME = "bert-base-uncased"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

print("Tokenizing dataset...")

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ============================================================
# 7. Load BERT Model
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading BERT model...")

# Red BERT load report hide karne ke liye stdout/stderr suppress kiya hai.
with open(os.devnull, "w") as f:
    with contextlib.redirect_stdout(f), contextlib.redirect_stderr(f):
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=4,
            id2label=id_to_label,
            label2id=label_to_id
        )

model.to(device)

print("Using device:", device)

# ============================================================
# 8. Evaluation Metrics
# ============================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")

    return {
        "accuracy": accuracy,
        "f1_score": f1
    }

# ============================================================
# 9. Training Arguments
# ============================================================

try:
    training_args = TrainingArguments(
        output_dir="./bert_ag_news_output",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=1,
        weight_decay=0.01,
        logging_steps=50,
        load_best_model_at_end=True,
        report_to="none",
        disable_tqdm=True
    )
except TypeError:
    training_args = TrainingArguments(
        output_dir="./bert_ag_news_output",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=1,
        weight_decay=0.01,
        logging_steps=50,
        load_best_model_at_end=True,
        report_to="none",
        disable_tqdm=True
    )

# ============================================================
# 10. Trainer Setup
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# ============================================================
# 11. Fine-Tune BERT
# ============================================================

print("\nTraining started. Please wait...")

trainer.train()

print("Training completed.")

# ============================================================
# 12. Evaluate Model
# ============================================================

print("\nEvaluating model...")

results = trainer.evaluate()

eval_accuracy = round(results.get("eval_accuracy", 0), 4)
eval_f1 = round(results.get("eval_f1_score", 0), 4)

print("\nEvaluation Results")
print("Accuracy:", eval_accuracy)
print("F1 Score:", eval_f1)

# ============================================================
# 13. Save Fine-Tuned Model
# ============================================================

trainer.save_model("./final_bert_ag_news_model")
tokenizer.save_pretrained("./final_bert_ag_news_model")

print("\nModel saved successfully!")

# ============================================================
# 14. Prediction Function
# ============================================================

def predict_news_topic(text):
    if text is None or text.strip() == "":
        return "No input", "0%", {
            "World": 0.0,
            "Sports": 0.0,
            "Business": 0.0,
            "Sci/Tech": 0.0
        }

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.nn.functional.softmax(outputs.logits, dim=1)[0]

    probabilities_dict = {
        id_to_label[i]: round(float(probabilities[i]), 4)
        for i in range(4)
    }

    best_class_id = int(torch.argmax(probabilities).item())
    best_label = id_to_label[best_class_id]
    confidence = round(float(probabilities[best_class_id]) * 100, 2)

    return best_label, str(confidence) + "%", probabilities_dict

# ============================================================
# 15. Test Prediction
# ============================================================

sample_text = "Apple launches new AI features for iPhone users"

print("\nSample Prediction:")
print(predict_news_topic(sample_text))

# ============================================================
# 16. Professional Gradio UI
# ============================================================

custom_css = """
.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
    font-family: 'Segoe UI', Arial, sans-serif !important;
    background: #f8fafc !important;
}

.header-box {
    background: linear-gradient(135deg, #0f172a, #1e40af);
    padding: 35px;
    border-radius: 22px;
    color: white;
    text-align: center;
    margin-bottom: 25px;
    box-shadow: 0 8px 22px rgba(15, 23, 42, 0.25);
}

.header-box h1 {
    font-size: 38px;
    margin-bottom: 8px;
}

.header-box p {
    font-size: 16px;
    opacity: 0.95;
}

.info-card {
    background: white;
    border: 1px solid #e2e8f0;
    padding: 18px;
    border-radius: 16px;
    text-align: center;
    margin-bottom: 16px;
    box-shadow: 0 4px 12px rgba(15, 23, 42, 0.06);
}

.info-card b {
    color: #0f172a;
    font-size: 16px;
}

.result-box {
    background: white;
    border-radius: 16px;
    padding: 10px;
}

.footer-note {
    text-align: center;
    color: #64748b;
    font-size: 13px;
    margin-top: 22px;
}

button {
    border-radius: 12px !important;
    font-weight: 700 !important;
}
"""

with gr.Blocks(
    title="BERT News Topic Classifier",
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="slate"
    ),
    css=custom_css
) as app:

    gr.HTML(
        """
        <div class="header-box">
            <h1>📰 BERT News Topic Classifier</h1>
            <p>Classify news headlines into World, Sports, Business, or Sci/Tech using a fine-tuned BERT model.</p>
        </div>
        """
    )

    with gr.Row():

        with gr.Column(scale=1):
            gr.HTML(
                f"""
                <div class="info-card">
                    <b>Model</b><br>
                    bert-base-uncased
                </div>

                <div class="info-card">
                    <b>Dataset</b><br>
                    AG News
                </div>

                <div class="info-card">
                    <b>Training Samples</b><br>
                    {TRAIN_SAMPLES}
                </div>

                <div class="info-card">
                    <b>Accuracy</b><br>
                    {eval_accuracy}
                </div>

                <div class="info-card">
                    <b>F1 Score</b><br>
                    {eval_f1}
                </div>
                """
            )

        with gr.Column(scale=2):
            user_input = gr.Textbox(
                label="Enter News Headline or News Text",
                placeholder="Example: Apple launches new AI features for iPhone users",
                lines=7
            )

            with gr.Row():
                predict_button = gr.Button("🚀 Classify News", variant="primary")
                clear_button = gr.Button("Clear")

            gr.Examples(
                examples=[
                    ["Apple launches new AI features for iPhone users"],
                    ["Pakistan wins cricket match after excellent bowling performance"],
                    ["Stock markets rise as investors expect strong company earnings"],
                    ["World leaders meet to discuss climate change policy"],
                    ["NASA announces a new mission to study Mars atmosphere"]
                ],
                inputs=user_input
            )

        with gr.Column(scale=1):
            gr.HTML("<div class='result-box'>")

            predicted_topic = gr.Textbox(
                label="Predicted Topic",
                interactive=False
            )

            confidence_score = gr.Textbox(
                label="Confidence",
                interactive=False
            )

            probability_output = gr.Label(
                label="Class Probabilities",
                num_top_classes=4
            )

            gr.HTML("</div>")

    predict_button.click(
        fn=predict_news_topic,
        inputs=user_input,
        outputs=[
            predicted_topic,
            confidence_score,
            probability_output
        ]
    )

    clear_button.click(
        fn=lambda: (
            "",
            "",
            "",
            {
                "World": 0.0,
                "Sports": 0.0,
                "Business": 0.0,
                "Sci/Tech": 0.0
            }
        ),
        inputs=None,
        outputs=[
            user_input,
            predicted_topic,
            confidence_score,
            probability_output
        ]
    )

    gr.HTML(
        """
        <div class="footer-note">
            Built with Hugging Face Transformers, PyTorch, Scikit-learn, and Gradio.
        </div>
        """
    )

# ============================================================
# 17. Launch App
# ============================================================

app.launch(
    inline=True,
    inbrowser=True,
    share=False
)

Loading AG News dataset...
Dataset loaded successfully
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
Sample: {'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}
Training samples: 800
Testing samples: 200
Loading tokenizer...
Tokenizing dataset...
Loading BERT model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Using device: cpu

Training started. Please wait...
